In [25]:
using Test

In [26]:
include("../RayTracing.jl")

Main.RayTracing

In [27]:
mi = 0
ma = 10
env_light = RayTracing.InfinteLight(
    RayTracing.Bounds3(RayTracing.Pnt3(mi), RayTracing.Pnt3(ma)), 
    RayTracing.RotateX(0.0), 
    RayTracing.Spectrum(1.0), 
    "../../ref/sky.exr"
)

Main.RayTracing.InfinteLight(Main.RayTracing.LightInfinite, Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 -0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; -0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; -0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 -0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), [1.0, 1.0, 1.0], Main.RayTracing.Distribution2D(Main.RayTracing.Distribution1D[Main.RayTracing.Distribution1D([5.0066897299937374e-5, 8.344479610759972e-5, 0.00011682264582683842, 0.00015020042682232444, 0.0001835781194587692, 0.0002169557041009361, 0.000250333161113652, 0.0002837104708618189, 0.0003170876137104254, 0.0003504645700245584  …  0.00028371047086182507, 0.00025033316111365463, 0.00021695570410093523, 0.0001835781194587842, 0.00015020042682233593, 0.00011682264582684636, 8.344479610760417e-5, 5.00668972999383e-5, 1.6688969039206713e-5, -1.6688969039196054e-5], [0.0, 4.413029

In [28]:
@test env_light.world_center ≈ RayTracing.Pnt3((mi+ma)/2)
@test env_light.world_radius ≈ sqrt(3 * ((ma-mi)/2)^2)

Test Passed

In [30]:
height, width = size(env_light.map)
_, max_coords = findmax(Float64.(RayTracing.Gray.(env_light.map)))

known_u = max_coords[2]/width
known_v = max_coords[1]/height
@test known_u ≈ 0.25
@test known_v ≈ 0.27783203125

known_u_idx = RayTracing._uv_map(known_u, width)
known_v_idx = RayTracing._uv_map(known_v, height)
@test RayTracing.Spectrum(env_light.map[known_v_idx, known_u_idx]) ≈ RayTracing.Spectrum(19008.0, 20320.0, 19680.0)

Test Passed

In [31]:
inter = RayTracing.Interaction(
    RayTracing.Pnt3(1,1,1),
    0.5,
    RayTracing.Vec3(1,1,1),
    RayTracing.Nml3(0,1,0)
)
sampled_li, wi, pdf_from_sample_li, vis, _, _ = RayTracing.sample_li(env_light, inter, RayTracing.Pnt2(0.5, 0.5))
@test sampled_li ≈ RayTracing.Spectrum(19008.0, 20320.0, 19680.0)

Test Passed

In [36]:
sampled_uv, pdf_from_sample_continuous = RayTracing.sample_continuous(env_light.pdf, RayTracing.Pnt2(0.5, 0.5))
@test isapprox(sampled_uv.x, known_u, rtol=3)
@test isapprox(sampled_uv.y, known_v, rtol=3)

Test Passed

In [39]:
pdf_from_sample_li

11681.4260584473

In [12]:
y, x = size(env_light.map)

(2048, 4096)

In [13]:
val, max_coords = findmax(Float64.(RayTracing.Gray.(env_light.map)))

(19856.0, CartesianIndex(569, 1024))

In [15]:
max_coords[2]/x, max_coords[1]/y

(0.25, 0.27783203125)

In [16]:
(u, v), pdf_val = RayTracing.sample_continuous(env_light.pdf, RayTracing.Pnt2(0.5, 0.5))

([0.25000225759775313, 0.27802136937694666], 176749.514924722)

In [17]:
function _fuck(a::Float64, b::Int64)::Int64
    return max(1,Int(trunc(a * b)))
end

_fuck (generic function with 1 method)

In [19]:
sample = env_light.map[_fuck(v, y), _fuck(u, x)]
sample.r, sample.g, sample.b

(Float16(1.9e4), Float16(2.032e4), Float16(1.968e4))

In [8]:
r = RayTracing.Pnt2(.5, .5)

2-element Main.RayTracing.Pnt2 with indices SOneTo(2):
 0.5
 0.5

# sample_li

In [9]:
uv, map_pdf = RayTracing.sample_continuous(env_light.pdf, r)
print("uv: ", uv, "\n")
print("map_pdf: ", map_pdf, "\n")

theta = uv.y * pi
phi = uv.x * 2 * pi
print("phi, theta: ", phi, ", ", theta, "\n")

cos_theta = RayTracing.cos(theta)
sin_theta = RayTracing.sin(theta)
sin_phi = RayTracing.sin(phi)
cos_phi = RayTracing.cos(phi)
wi = env_light.light_to_world(RayTracing.Vec3(sin_theta * cos_phi, sin_theta * sin_phi, cos_theta))
print("wi: ", wi, "\n")

map_pdf /= (2 * pi * pi * sin_theta)
print("map_pdf: ", map_pdf, "\n")

x, y = size(env_light.map)
uu = RayTracing.Int(RayTracing.trunc(uv.x * x)+1)
vv = RayTracing.Int(RayTracing.trunc(uv.x * x)+1)
radiance = env_light.map[uu, vv]
print("radiance: ", radiance.r, ", ", radiance.g, ", ", radiance.b, "\n")

uv: [0.2500023667131565, 0.2780281699157606]
map_pdf: 171917.03873114358


phi, theta: 1.5708111972922278, 0.8734512560983683
wi: 

[-1.1398977381377058e-5, 0.7665498420554099, 0.6421848172565974]
map_pdf: 11361.842958503961
radiance: 0.04074, 0.09467, 0.1324


# pdf_li

In [11]:
wi = env_light.world_to_light(wi)
theta = RayTracing.spherical_theta(wi)
phi = RayTracing.spherical_phi(wi)
sin_theta = RayTracing.sin(theta)
(sin_theta == 0.0) && return 0.0

u_idx = phi / 2pi
v_idx = theta / pi
print("uv: ", u_idx, ", ", v_idx, "\n")

pdf_val = RayTracing.pdf(env_light.pdf, RayTracing.Pnt2(u_idx, v_idx))
print("pdf_map: ", pdf_val, "\n")
print("pdf_map: ", pdf_val / (2 * pi * pi * sin_theta), "\n")

uv: 0.2500023667131565, 0.2780281699157606
pdf_map: 171917.03873114358
pdf_map: 11361.842958503961
